<a href="https://colab.research.google.com/github/harshitasdev8/Myeloid-Project-Oda-Lab/blob/PDAC_MultiGenes/PDAC_Working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
"""
PDAC Myeloid Score — Survival Analysis Pipeline (v2: composite scores)
========================================================================
Data source: UCSC Xena, TCGA Pancreatic Cancer (PAAD)

INSTRUCTIONS FOR YOU:
1. Change RAW_PATH below to wherever your new TSV file is saved on your computer.
2. Make sure you have 'lifelines' and 'pandas' installed. If not, run this first
   in a notebook cell:  !pip install lifelines pandas
3. Run all cells / run the whole script top to bottom.
4. All results print to the screen AND get saved as CSV files you can open in
   a spreadsheet (see the bottom of the script for exact filenames).

GENES USED, GROUPED INTO AXES:
- Macrophage abundance : CD68, CD14
- M1 (pro-inflammatory) : CD80, CD86, HLA-DRA, NOS2
- M2 (suppressive)      : CD163, MRC1, MSR1
- CD8 T cells           : CD8A, CD8B
- CD40 (gate)           : CD40

If you weren't able to pull one of these genes in Xena (e.g. HLA-DRA didn't
match), just delete it from the relevant list below — the script will still
run fine with fewer genes in that axis.
"""

"\nPDAC Myeloid Score — Survival Analysis Pipeline (v2: composite scores)\n========================================================================\nData source: UCSC Xena, TCGA Pancreatic Cancer (PAAD)\n\nINSTRUCTIONS FOR YOU:\n1. Change RAW_PATH below to wherever your new TSV file is saved on your computer.\n2. Make sure you have 'lifelines' and 'pandas' installed. If not, run this first\n   in a notebook cell:  !pip install lifelines pandas\n3. Run all cells / run the whole script top to bottom.\n4. All results print to the screen AND get saved as CSV files you can open in\n   a spreadsheet (see the bottom of the script for exact filenames).\n\nGENES USED, GROUPED INTO AXES:\n- Macrophage abundance : CD68, CD14\n- M1 (pro-inflammatory) : CD80, CD86, HLA-DRA, NOS2\n- M2 (suppressive)      : CD163, MRC1, MSR1\n- CD8 T cells           : CD8A, CD8B\n- CD40 (gate)           : CD40\n\nIf you weren't able to pull one of these genes in Xena (e.g. HLA-DRA didn't\nmatch), just delete it from 

In [42]:
pip install lifelines

In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
import pandas as pd
from lifelines import CoxPHFitter
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------------------------
# STEP 0: EDIT THIS PATH to point to your new TSV file
# ---------------------------------------------------------------------------
RAW_PATH = "/content/denseDataOnlyDownload_age+tumorstage.tsv"   # <-- change to your new file path

# ---------------------------------------------------------------------------
# STEP 1: Define which genes and clinical vars belong to the analysis
# ---------------------------------------------------------------------------
ABUNDANCE_GENES = ["CD68", "CD14"]
M1_GENES        = ["CD86", "HLA-DRA"]
M2_GENES        = ["CD163", "MRC1", "MSR1"]
CD8_GENES       = ["CD8A", "CD8B"]
CD40_GENE       = ["CD40"]

# Adding age and stage to our list of variables to keep
CLINICAL_VARS = ["age_at_initial_pathologic_diagnosis", "pathologic_stage"]

ALL_GENES = ABUNDANCE_GENES + M1_GENES + M2_GENES + CD8_GENES + CD40_GENE
ALL_COVARIATES = ALL_GENES + CLINICAL_VARS

In [45]:


import os

# Define the Google Drive folder path
drive_output_dir = "/content/drive/MyDrive/Colab Notebooks/PDAC_MultiGenes_Data"

# Create the directory if it doesn't exist
os.makedirs(drive_output_dir, exist_ok=True)

In [46]:
# ---------------------------------------------------------------------------
# STEP 2: Load raw data
# ---------------------------------------------------------------------------
df = pd.read_csv(RAW_PATH, sep="\t")

# Keep only columns that actually exist in your file
ALL_GENES = [g for g in ALL_GENES if g in df.columns]
CLINICAL_VARS = [c for c in CLINICAL_VARS if c in df.columns]

print("Genes found:", ALL_GENES)
print("Clinical variables found:", CLINICAL_VARS)

Genes found: ['CD68', 'CD14', 'CD86', 'HLA-DRA', 'CD163', 'MRC1', 'MSR1', 'CD8A', 'CD8B', 'CD40']
Clinical variables found: ['age_at_initial_pathologic_diagnosis', 'pathologic_stage']


In [47]:
# ---------------------------------------------------------------------------
# STEP 3: Clean — keep only primary tumor samples and handle stage/age
# ---------------------------------------------------------------------------
df["sample_type_code"] = df["sample"].str.split("-").str[-1]
df_tumor = df[df["sample_type_code"] == "01"].copy()

# Basic cleaning for Age (ensure numeric)
# Variable name from STEP 1: age_at_initial_pathologic_diagnosis
if "age_at_initial_pathologic_diagnosis" in df_tumor.columns:
    df_tumor["age"] = pd.to_numeric(df_tumor["age_at_initial_pathologic_diagnosis"], errors='coerce')

# Basic cleaning for Stage (simplified to numeric or ordinal if possible)
def simplify_stage(stage_str):
    if pd.isna(stage_str): return None
    s = str(stage_str).lower()
    if 'iv' in s: return 4
    if 'iii' in s: return 3
    if 'ii' in s: return 2
    if 'i' in s: return 1
    return None

# Using the actual column name found in your dataset: pathologic_stage
if "pathologic_stage" in df_tumor.columns:
    df_tumor["stage_numeric"] = df_tumor["pathologic_stage"].apply(simplify_stage)
elif "tumor_stage" in df_tumor.columns:
    df_tumor["stage_numeric"] = df_tumor["tumor_stage"].apply(simplify_stage)

# Drop rows missing critical survival, gene, or clinical data
# We check if the columns exist before dropping to avoid KeyError
required_cols = [g for g in ALL_GENES]
if "age" in df_tumor.columns: required_cols.append("age")
if "stage_numeric" in df_tumor.columns: required_cols.append("stage_numeric")

df_tumor = df_tumor.dropna(subset=required_cols)
print(f"Samples after dropping missing gene/clinical data: {df_tumor.shape[0]}")

Samples after dropping missing gene/clinical data: 177


In [48]:
# ---------------------------------------------------------------------------
# STEP 4: Build survival time + event indicator
# ---------------------------------------------------------------------------
df_tumor["event"] = (df_tumor["vital_status"] == "DECEASED").astype(int)
df_tumor["time"] = df_tumor["days_to_death"].fillna(df_tumor["days_to_last_followup"])
assert df_tumor["time"].isna().sum() == 0, "Some patients have no survival time — check raw data"

print(f"\nFinal analysis cohort: {df_tumor.shape[0]} patients "
      f"({df_tumor['event'].sum()} deaths, {(df_tumor['event']==0).sum()} censored)")


Final analysis cohort: 177 patients (93 deaths, 84 censored)


In [49]:
 #---------------------------------------------------------------------------
# STEP 5: Build composite scores
#          Each composite = simple average of its genes' expression values.
#          (Gene expression here is already log2-scale, so averaging is standard.)
# ---------------------------------------------------------------------------
if ABUNDANCE_GENES:
    df_tumor["abundance_score"] = df_tumor[ABUNDANCE_GENES].mean(axis=1)
if M1_GENES:
    df_tumor["M1_score"] = df_tumor[M1_GENES].mean(axis=1)
if M2_GENES:
    df_tumor["M2_score"] = df_tumor[M2_GENES].mean(axis=1)
if CD8_GENES:
    df_tumor["CD8_score"] = df_tumor[CD8_GENES].mean(axis=1)
if CD40_GENE:
    df_tumor["CD40_score"] = df_tumor[CD40_GENE].mean(axis=1)  # just CD40 itself

# M1:M2 ratio (log-scale subtraction = ratio)
if M1_GENES and M2_GENES:
    df_tumor["M1_M2_ratio"] = df_tumor["M1_score"] - df_tumor["M2_score"]

# M2:CD8 "suppression wall" ratio
if M2_GENES and CD8_GENES:
    df_tumor["M2_CD8_ratio"] = df_tumor["M2_score"] - df_tumor["CD8_score"]

COMPOSITE_VARS = [c for c in
                   ["abundance_score", "M1_score", "M2_score", "CD8_score",
                    "CD40_score", "M1_M2_ratio", "M2_CD8_ratio"]
                   if c in df_tumor.columns]


In [50]:
# ---------------------------------------------------------------------------
# STEP 6b: Updated Multivariate Cox model — adjusting for Age and Stage
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("MULTIVARIATE COX MODEL (Genes + Age + Stage)")
print("=" * 70)

# We include the gene scores PLUS clinical variables
analysis_vars = [v for v in ["abundance_score", "M1_score", "M2_score", "CD8_score", "CD40_score", "age", "stage_numeric"]
                 if v in df_tumor.columns]

cph_multi = CoxPHFitter()
cph_multi.fit(df_tumor[analysis_vars + ["time", "event"]], duration_col="time", event_col="event")
print(cph_multi.summary[["coef", "exp(coef)", "p"]].to_string())
print(f"\nConcordance index: {cph_multi.concordance_index_:.3f}")

# ---------------------------------------------------------------------------
# STEP 7: Save everything
# ---------------------------------------------------------------------------
df_tumor.to_csv("pdac_clean_with_clinical_and_scores.csv", index=False)
cph_multi.summary.to_csv("pdac_multivariate_clinical_adjusted_results.csv")


MULTIVARIATE COX MODEL (Genes + Age + Stage)
                     coef  exp(coef)         p
covariate                                     
abundance_score -0.354853   0.701276  0.197716
M1_score         0.153667   1.166102  0.547754
M2_score         0.092458   1.096867  0.564462
CD8_score       -0.179128   0.835999  0.088000
CD40_score       0.329989   1.390953  0.030254
age              0.029156   1.029585  0.007831
stage_numeric    0.243483   1.275684  0.235591

Concordance index: 0.601


### Consolidating results into an Excel file

In [51]:
import shutil
import os
import pandas as pd

# Define paths
local_excel_path = "pdac_all_results_temp.xlsx"
filename = "pdac_all_results.xlsx"

print("Generating fresh Excel file from current session data...")

# 1. Save to Excel using ONLY current memory variables to avoid 'carrying over' old data
with pd.ExcelWriter(local_excel_path, engine='openpyxl') as writer:
    # Sheet 1: The full cleaned dataset (includes Age and Stage)
    if 'df_tumor' in globals():
        df_tumor.to_excel(writer, sheet_name='Cleaned Data', index=False)

    # Sheet 2: Univariate Results (from the df_univariate variable)
    if 'df_univariate' in globals():
        df_univariate.to_excel(writer, sheet_name='Univariate Results', index=False)

    # Sheet 3: Multivariate Adjusted (Genes + Age + Stage)
    if 'cph_multi' in globals():
        cph_multi.summary.to_excel(writer, sheet_name='Multivariate Adjusted')

# 2. Robust Drive Saving
try:
    if not os.path.exists('/content/drive/MyDrive'):
        print("Warning: Drive not mounted. Run the Drive Mount cell above.")
    else:
        os.makedirs(drive_output_dir, exist_ok=True)
        dest_path = os.path.join(drive_output_dir, filename)

        # Overwrite destination with the fresh local copy
        shutil.copy2(local_excel_path, dest_path)
        print(f"Successfully saved fresh results to Drive: {dest_path}")
except Exception as e:
    print(f"Drive save failed: {e}")
    print(f"You can still download the fresh file '{local_excel_path}' from the file browser on the left.")

Generating fresh Excel file from current session data...
Successfully saved fresh results to Drive: /content/drive/MyDrive/Colab Notebooks/PDAC_MultiGenes_Data/pdac_all_results.xlsx


In [52]:
from lifelines import CoxPHFitter

def fit_model(vars_to_use, name):
    cph = CoxPHFitter()
    cph.fit(
        df_tumor[vars_to_use + ["time", "event"]],
        duration_col="time",
        event_col="event"
    )
    print(f"{name}: C-index = {cph.concordance_index_:.3f}")
    return cph

# Genes only
gene_vars = [
    "abundance_score",
    "M1_score",
    "M2_score",
    "CD8_score",
    "CD40_score"
]

# Clinical only
clinical_vars = [
    "age",
    "stage_numeric"
]

fit_model(gene_vars, "Genes only")
fit_model(clinical_vars, "Age + Stage")
fit_model(gene_vars + clinical_vars, "Genes + Age + Stage")

Genes only: C-index = 0.558
Age + Stage: C-index = 0.565
Genes + Age + Stage: C-index = 0.601


<lifelines.CoxPHFitter: fitted with 177 total observations, 84 right-censored observations>